# Pandas Data Cleaning: การทำความสะอาดข้อมูลด้วย Pandas ตั้งแต่พื้นฐานถึงขั้นสูง

Notebook นี้ครอบคลุมเทคนิคการทำความสะอาดข้อมูล (Data Cleaning) ในหลายมิติ โดยมุ่งให้ผู้เรียนเข้าใจทั้ง "วิธีการ"
"เหตุผล" และ "สถานการณ์ที่เหมาะสม" ในการเลือกใช้แต่ละเทคนิค

---

In [1]:
import pandas as pd
import numpy as np
import re

## Part 1: Regular Expressions (Regex) โดยละเอียด

ก่อนนำ Regex ไปใช้ร่วมกับ Pandas ควรทำความเข้าใจพื้นฐานของ **Raw String** และ **โมดูล `re`** ใน Python เนื่องจากเป็นพื้นฐานสำคัญของหัวข้อถัดไป

### 1.1 Raw String (r"")

ใน Python เครื่องหมาย Backslash (`\`) เป็นอักขระพิเศษ (Escape Character) เช่น `\n` หมายถึงการขึ้นบรรทัดใหม่
แต่ใน Regex มีการใช้ `\` จำนวนมาก (เช่น `\d`, `\w`) จึงต้องระบุให้ Python **ไม่แปลความหมายของ Backslash** โดยการใส่ `r` นำหน้า String

In [2]:
# Normal String: \ จะถูกตีความว่าเป็น escape character
path = "C:\desktop\Ravikanth" 
print("Normal String:", path) 
# สังเกตว่า \d อาจจะหายไป หรือ \r กลายเป็น carriage return

# Raw String: \ เป็นแค่ตัวอักษรธรรมดา
path_raw = r"C:\desktop\Ravikanth"
print("Raw String:   ", path_raw)

Normal String: C:\desktop\Ravikanth
Raw String:    C:\desktop\Ravikanth


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\pc\AppData\Local\Temp\ipykernel_21908\1691487632.py:2: SyntaxWarning: invalid escape sequence '\d'
  path = "C:\desktop\Ravikanth"


### 1.2 คำสั่งหลักของโมดูล `re`

Python มีคำสั่งหลัก 3 คำสั่งสำหรับการใช้งาน Regex ดังนี้

1.  **`re.match(pattern, string)`**: ตรวจสอบ **เฉพาะตำแหน่งเริ่มต้น** ของข้อความ (หากตำแหน่งเริ่มต้นไม่ตรงกับรูปแบบ จะคืนค่า `None`)
2.  **`re.search(pattern, string)`**: ค้นหา **ตำแหน่งแรกที่พบ** (First Occurrence) ในข้อความ และหยุดค้นหาเมื่อพบ
3.  **`re.findall(pattern, string)`**: ค้นหา **ทุกตำแหน่งที่พบ** (All Occurrences) และคืนค่าเป็น List

**ข้อแนะนำ:** `re.findall()` เป็นคำสั่งที่ใช้บ่อยที่สุด เนื่องจากครอบคลุมและสะดวกต่อการดึงข้อมูล

In [3]:
text = "Kaggle is the largest data science community of India"

# 1. re.match() - หาคำว่า Kaggle ที่ต้นประโยค (เจอ)
print("Match 'Kaggle':", re.match('Kaggle', text))

# 2. re.match() - หาคำว่า largest ที่ต้นประโยค (ไม่เจอ)
print("Match 'largest':", re.match('largest', text))

Match 'Kaggle': <re.Match object; span=(0, 6), match='Kaggle'>
Match 'largest': None


In [4]:
# 3. re.search() - หาคำว่า largest (เจอตรงกลาง)
search_res = re.search('largest', text)
print("Search 'largest':", search_res)
print("Found at:", search_res.span()) # บอกพิกัด

Search 'largest': <re.Match object; span=(14, 21), match='largest'>
Found at: (14, 21)


In [5]:
text2 = "Andrew NG founded Coursera. He also founded deeplearning.ai"

# 4. re.findall() - หาคำว่า founded ทั้งหมด
print("Findall 'founded':", re.findall('founded', text2))

Findall 'founded': ['founded', 'founded']


### 1.3 Special Sequences (รหัสพิเศษของ Regex)

Special Sequences เป็นรูปแบบย่อที่ช่วยให้เขียน Regex ได้กระชับและมีประสิทธิภาพ

| Code | ความหมาย | ตัวอย่าง |
| :--- | :--- | :--- |
| `\d` | ตัวเลข (0-9) | `\d+` ดึงตัวเลขที่อยู่ติดกันทั้งหมด |
| `\D` | อักขระที่ **ไม่ใช่** ตัวเลข | `\D+` ดึงตัวอักษรทั้งหมด |
| `\w` | ตัวอักษรและตัวเลข (Word) | `\w+` ดึงคำ (ภาษาอังกฤษและตัวเลข) |
| `\W` | อักขระที่ **ไม่ใช่** Word (สัญลักษณ์) | `\W+` ดึงช่องว่างหรือเครื่องหมาย |
| `\s` | ช่องว่าง (Space) | `\s` ตรงกับช่องว่าง 1 ตัว |
| `\b` | ขอบเขตของคำ (Boundary) | `\best` คำที่ลงท้ายด้วย est |
| `.` | อักขระใดก็ได้ (Wildcard) | `a.c` ตรงกับ abc, acc, a@c |

In [6]:
str_data = "2 million monthly visits in Jan'19."

# \d : หาเลขทีละตัว
print("\\d (Digits):", re.findall(r"\d", str_data))

# \d+ : หาเลขที่เป็นกลุ่มก้อน (โลภมาก)
print("\\d+ (Number chunks):", re.findall(r"\d+", str_data))

# \D : หาที่ไม่ใช่เลข
print("\\D (Non-digits):", re.findall(r"\D+", str_data))

# \w+ : หาคำ (รวมตัวเลข)
print("\\w+ (Words):", re.findall(r"\w+", str_data))

\d (Digits): ['2', '1', '9']
\d+ (Number chunks): ['2', '19']
\D (Non-digits): [" million monthly visits in Jan'", '.']
\w+ (Words): ['2', 'million', 'monthly', 'visits', 'in', 'Jan', '19']


### 1.4 Metacharacters (อักขระควบคุม)

- `^` : ตำแหน่งเริ่มต้น (Start)
- `$` : ตำแหน่งสิ้นสุดบรรทัด (End)
- `.` : อักขระใดก็ได้
- `*` : ปรากฏ 0 ครั้งขึ้นไป
- `+` : ปรากฏ 1 ครั้งขึ้นไป
- `?` : ปรากฏ 0 หรือ 1 ครั้ง
- `|` : หรือ (OR)

In [ ]:
example = "easy easssy eay ey"

# * (0 หรือมากกว่า): ea ตามด้วย s กี่ตัวก็ได้ (หรือไม่มี s) แล้วจบด้วย y
# easy (s=1), easssy (s=3), eay (s=0)
print("eas*y:", re.findall(r"eas*y", example))

# + (1 หรือมากกว่า): ต้องมี s อย่างน้อย 1 ตัว
# easy, easssy (eay ไม่ติด เพราะไม่มี s)
print("eas+y:", re.findall(r"eas+y", example))

# ? (0 หรือ 1): มี s ได้มากสุด 1 ตัว
# easy, eay (easssy ไม่ติด เพราะ s>1)
print("eas?y:", re.findall(r"eas?y", example))

### 1.5 กรณีศึกษาการใช้งานจริง (Real World Use Cases)

#### Case 1: Extract Email IDs
โจทย์: ดึงอีเมลทั้งหมดจากข้อความ

In [ ]:
email_text = 'Send mail to rohan.1997@gmail.com, smith_david34@yahoo.com about meeting @2PM'

# [a-zA-Z0-9._-]+  : ชื่อเมล (ตัวอักษร, ตัวเลข, จุด, ขีดล่าง, ขีดกลาง)
# @                 : เครื่องหมาย @
# \w+               : ชื่อ domain (gmail, yahoo)
# \.com             : .com (ต้องมี \ เพราะ . คือ wildcard)

emails = re.findall(r'[a-zA-Z0-9._-]+@\w+\.com', email_text)
print("Emails found:", emails)

#### Case 2: Extract Dates
โจทย์: ดึงวันที่ซึ่งมีรูปแบบการเขียนหลากหลาย

In [ ]:
date_text = "London Olympic 2012 was held from 27 July 2012 to 12 August 2012."

# \d{2}       : วันที่ 2 ตัว
# .           : ตัวคั่นอะไรก็ได้ (space)
# \w{3,10}    : เดือน (ภาษาอังกฤษ 3-10 ตัวอักษร)
# .           : ตัวคั่น
# \d{4}       : ปี 4 ตัว

dates = re.findall(r'\d{2}.\w{3,10}.\d{4}', date_text)
print("Dates found:", dates)

#### Case 3: Titanic Title Extraction
โจทย์: ในชุดข้อมูล Titanic ชื่อผู้โดยสารมีคำนำหน้า (Mr., Mrs., Miss.) ต้องการดึงคำนำหน้าออกมาเพื่อวิเคราะห์

รูปแบบข้อมูล: `Names, Title. Firstname` → ต้องการคำที่อยู่หลังเครื่องหมาย `,` และก่อนเครื่องหมาย `.`

In [ ]:
import kagglehub
import os

# Load Titanic Dataset (ถ้ายังไม่มี)
path = kagglehub.dataset_download("rahulsah06/titanic")
titanic = pd.read_csv(os.path.join(path, "train.csv"))

# ดูตัวอย่างชื่อ
print(titanic['Name'].head())

In [ ]:
# วิธีใช้ Regex ดึง Title
# regex=([A-Za-z]+)\. 
# [A-Za-z]+ : ตัวอักษรภาษาอังกฤษกี่ตัวก็ได้
# \.        : ต้องจบด้วยจุด (.) 
# ()        : วงเล็บคือ Group ที่เราจะดึงมา (ไม่เอาจุดมาด้วย)

titanic['Title'] = titanic['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

print("\nUnique Titles Found:")
print(titanic['Title'].value_counts())

## Part 2: การเลือกข้อมูลด้วย `.loc` และ `.iloc`

หัวใจสำคัญของการจัดการข้อมูล (Data Wrangling) คือการเลือกข้อมูล (Selection) ได้อย่างถูกต้องแม่นยำ
Pandas มีคำสั่งหลัก 2 คำสั่งที่ต้องใช้ให้คล่อง ได้แก่ `.loc` และ `.iloc`

### ความแตกต่างหลัก (Key Difference)
- **`.loc` (Label-based):** อ้างอิงจาก **ชื่อ** ของแถว (Index) หรือคอลัมน์
- **`.iloc` (Integer-based):** อ้างอิงจาก **ตำแหน่งลำดับ** (0, 1, 2, ...) โดยไม่พิจารณาชื่อ

In [ ]:
# สร้าง DataFrame ตัวอย่าง
df_select = pd.DataFrame({
    'A': [10, 20, 30, 40],
    'B': [100, 200, 300, 400],
    'C': ['cat', 'dog', 'bird', 'fish']
}, index=['row1', 'row2', 'row3', 'row4']) # กำหนด index เป็นชื่อ

df_select

### 2.1 การใช้งาน `.loc` (Label-based)

**รูปแบบ:** `df.loc[row_label, column_label]`

1.  **Single Label:** เลือก 1 แถว 1 คอลัมน์
2.  **List of Labels:** เลือกหลายรายการพร้อมกัน
3.  **Slice (ช่วงข้อมูล):** เลือกตั้งแต่ตำแหน่งหนึ่งถึงอีกตำแหน่งหนึ่ง (ข้อควรระวัง: `.loc` **รวมตำแหน่งสุดท้าย** ด้วย กล่าวคือ `start:stop` จะรวม `stop`)

In [ ]:
# 1. Select specific cell
print("1. Value at row2, col B:", df_select.loc['row2', 'B'])

# 2. Select list of rows/cols
print("\n2. List selection:\n", df_select.loc[['row1', 'row3'], ['A', 'C']])

# 3. Slicing (สังเกตว่า 'row3' จะติดมาด้วย)
print("\n3. Slicing row1 to row3:\n", df_select.loc['row1':'row3', 'A':'B'])

### 2.2 การใช้งาน `.iloc` (Position-based)

**รูปแบบ:** `df.iloc[row_position, column_position]`

เหมาะสำหรับกรณีที่ไม่ทราบชื่อคอลัมน์ หรือต้องการเลือกแถวแรก/แถวสุดท้าย

In [ ]:
# 1. Select by position
print("1. Row 0 (First row):\n", df_select.iloc[0])

# 2. Slicing (ไม่รวมตัวสุดท้าย เหมือน Python list ปกติ)
# 0:2 ได้แค่ 0, 1 (row1, row2)
print("\n2. First 2 rows:\n", df_select.iloc[0:2])

# 3. Negative Indexing (ตัวเทพ)
# เลือก 2 แถวสุดท้าย, และคอลัมน์สุดท้าย
print("\n3. Last 2 rows, last column:\n", df_select.iloc[-2:, -1])

### 2.3 Boolean Indexing ด้วย `.loc`

เป็นวิธีที่ใช้บ่อยที่สุดในการทำงานจริง คือการเลือกข้อมูลตาม **เงื่อนไข**

In [ ]:
# เลือกแถวที่ค่า A มากกว่า 20
mask = df_select['A'] > 20
print("Condition Mask:\n", mask)

# ใช้ mask ในการ filter
result = df_select.loc[mask, ['B', 'C']]
print("\nResult (A > 20):\n", result)

# เขียนแบบย่อ (Oneliner)
print("\nOneliner (A > 20 and C != 'bird'):\n", 
      df_select.loc[(df_select['A'] > 20) & (df_select['C'] != 'bird')])

### 2.4 การแก้ไขค่าข้อมูล (Setting Values)

สามารถใช้ `.loc` เพื่อแก้ไขค่าข้อมูลเฉพาะตำแหน่งที่ต้องการได้

**ข้อควรระวัง:** ควรหลีกเลี่ยงการเขียน `df[condition]['col'] = val` (Chained Assignment) และใช้ `.loc` แทนเสมอ

In [ ]:
# แก้ค่า C ของแถวที่ A = 10 ให้เป็น 'kitten'
df_select.loc[df_select['A'] == 10, 'C'] = 'kitten'
df_select

## Part 2.5: แบบฝึกปฏิบัติการเลือกข้อมูลจริง (Titanic)

ส่วนนี้นำ `.loc` และ `.iloc` มาใช้กับชุดข้อมูลจริง **Titanic** เพื่อให้เข้าใจพื้นฐานอย่างถ่องแท้

### Concept 1: การเลือกด้วย `[]` เทียบกับ `.loc` / `.iloc`

คำถามที่พบบ่อยคือ เหตุใดจึงต้องใช้ `.loc` ในเมื่อสามารถใช้ `[]` ได้

- **`df['Age']` (Square Brackets):** ใช้สำหรับเลือก **คอลัมน์** เป็นหลัก (ใช้งานง่ายและรวดเร็ว แต่ยืดหยุ่นน้อย)
- **`df.loc[...]` / `df.iloc[...]` (Accessors):** เป็นการเลือกแบบ **ระบุพิกัด** (Coordinate) ได้ทั้ง [แถว, คอลัมน์]
    - เปรียบเทียบกับการระบุที่อยู่:
        - `.loc`: ระบุด้วย **ชื่อถนน** (Name: Allen, Col: Age)
        - `.iloc`: ระบุด้วย **พิกัด GPS** (Row: 5, Col: 3)

**หลักการ:** `df.loc[แถวที่ต้องการ, คอลัมน์ที่ต้องการ]`

In [ ]:
# Install kagglehub if not already installed
!pip install kagglehub

In [ ]:
import kagglehub
import os

# Load Dataset Titanic
path = kagglehub.dataset_download("rahulsah06/titanic")
titanic = pd.read_csv(os.path.join(path, "train.csv"))
titanic.head(3)

### 1. การเลือกคอลัมน์อย่างง่าย (`[]`)

- เลือก 1 คอลัมน์ → ใช้ `['ชื่อ']` → ได้ผลลัพธ์เป็น **Series** (ข้อมูลหนึ่งมิติ)
- เลือกหลายคอลัมน์ → ใช้ `[['ชื่อ1', 'ชื่อ2']]` (วงเล็บซ้อน) → ได้ผลลัพธ์เป็น **DataFrame** (ตาราง)

In [ ]:
# แบบที่ 1: เลือกคอลัมน์เดียว (Series)
age_series = titanic['Age']

# แบบที่ 2: เลือกหลายคอลัมน์ (DataFrame)
subset_df = titanic[['Age', 'Pclass', 'Fare']]

print("Type 1:", type(age_series))
print("Type 2:", type(subset_df))
subset_df.head(3)

### 2. การเลือกด้วย `.iloc` (Integer Location)

ตัวอักษร **"i"** ย่อมาจาก **"Integer"** (จำนวนเต็ม)
ใช้เมื่อต้องการอ้างอิง **ลำดับที่** ของข้อมูล โดยไม่พิจารณาชื่อ Index

**รูปแบบ:** `df.iloc[row_index, col_index]`

In [ ]:
# ตัวอย่าง: อยากได้ "แถวที่ 3" (Index 3)
row_3 = titanic.iloc[3]
print("---> Row 3 Data <___")
print(row_3)

# ตัวอย่าง: อยากได้ "3 แถวแรก" (0, 1, 2)
# ใช้ Slicing [Start : Stop] (ตัว Stop ไม่ถูกรวม)
first_3_rows = titanic.iloc[0:3]
print("\n---> First 3 Rows <---")
first_3_rows

In [ ]:
# Advance: เลือกทั้งแถวและคอลัมน์ด้วยตัวเลข
# อยากได้แถวที่ 10-25
# และเอาเฉพาะคอลัมน์ที่ 3-5 (Name, Sex, Age...)
titanic.iloc[9:25, 2:5]

### 3. การเลือกด้วย `.loc` (Label Location)

**"Label"** หมายถึง **ชื่อป้ายกำกับ**
ใช้เมื่อทราบ **ชื่อ Index** หรือ **ชื่อคอลัมน์**

**ข้อควรระวัง:** การ Slice ด้วย `.loc` เช่น `'A':'C'` จะ **รวมตำแหน่งสุดท้าย ('C') ด้วย** ซึ่งต่างจาก Slicing ปกติของ Python

In [ ]:
# เพื่อให้เห็นภาพชัด เราจะสร้าง DataFrame ใหม่ที่ตั้ง "ชื่อคน" เป็น Index
titanic_named = titanic.set_index("Name")
titanic_named.head(3)

In [ ]:
# ทีนี้เราเลือกข้อมูลจาก "ชื่อคน" ได้เลย!
# อยากรู้ข้อมูลของ 'Mr. William Henry Allen'
try:
    person = titanic_named.loc['Allen, Mr. William Henry']
    print(person)
except KeyError:
    print("หาชื่อไม่เจอ ลองเลือกชื่อแรกจาก .head() มาแทนนะครับ")

In [ ]:
# เลือกหลายคน และเลือกเฉพาะคอลัมน์ที่สนใจ
# สังเกตว่าเราใช้ "ชื่อ" ทั้งหมดใน []
names_to_find = [
    'Braund, Mr. Owen Harris',
    'Heikkinen, Miss. Laina'
]
cols_to_show = ['Pclass', 'Fare', 'Embarked']

titanic_named.loc[names_to_find, cols_to_show]

### 4. Filtering: การกรองข้อมูลด้วยเงื่อนไข

การกรองข้อมูลเป็นงานที่ Data Scientist ทำบ่อยที่สุด เปรียบเสมือนการตั้งคำถามกับข้อมูล

**โจทย์:** ค้นหาผู้รอดชีวิต (Survived = 1) ที่เป็นผู้โดยสารชั้น 1 (Pclass = 1) และจ่ายค่าโดยสารมากกว่า 200

สามารถทำได้ 2 วิธี ดังนี้

In [ ]:
# วิธีที่ 1: Boolean Indexing (ท่ามาตรฐาน)
# ข้อดี: ยืดหยุ่น custom ได้เยอะ
# ข้อเสีย: ต้องพิมพ์ชื่อ df ซ้ำๆ และต้องใส่วงเล็บ () ครบทุกเงื่อนไข

condition = (titanic['Survived'] == 1) & \
            (titanic['Pclass'] == 1) & \
            (titanic['Fare'] > 200)

titanic[condition]

In [ ]:
# วิธีที่ 2: .query() (ท่าง่าย อ่านรู้เรื่อง)
# ข้อดี: เขียนเหมือน SQL อ่านแล้วเข้าใจเลย
# ข้อเสีย: ใช้ตัวแปรภายนอกต้องมี @ นำหน้า

titanic.query('Survived == 1 and Pclass == 1 and Fare > 200')

## Part 3: การจัดการข้อความขั้นสูง (Advanced String Manipulation)

นอกเหนือจาก `.strip()` และ `.replace()` ยังมีเทคนิคอื่นที่นิยมใช้ในงาน Data Science ดังนี้

In [ ]:
df = pd.DataFrame({'Full_Name': ['John Doe', 'Jane Smith', 'Bob Lee']})

# 3.1 Splitting Strings
# แยกชื่อ-นามสกุล ออกเป็น Columns ใหม่
# expand=True จะแตกเป็น DataFrame ให้เลย (ถ้า False จะเป็น List ใน cell เดียว)
names = df['Full_Name'].str.split(' ', expand=True)
names.columns = ['First', 'Last']
df = pd.concat([df, names], axis=1)
df

In [ ]:
# 3.2 String Concatenation (ต่อคำ)
# เอานามสกุลขึ้นก่อน คั่นด้วย comma
df['Last_First'] = df['Last'].str.cat(df['First'], sep=', ')
df

In [ ]:
# 3.3 Advanced Replace (ใช้ Dictionary)
# แทนที่หลายคำพร้อมกัน
df['Status'] = ['Process', 'Done', 'Wip']
replacements = {
    'Process': 'In Progress',
    'Wip': 'In Progress',
    'Done': 'Completed'
}
df['Status_Clean'] = df['Status'].replace(replacements)
df

## Part 4: วิธีการเลือกข้อมูลแบบพิเศษ (`.query`, `.select_dtypes`)

เป็นทางเลือกในการเลือกข้อมูลที่อ่านง่ายและมีประสิทธิภาพ

In [ ]:
data = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 40],
    'Department': ['HR', 'IT', 'Finance', 'IT'],
    'Salary': [50000, 60000, 55000, 70000]
})

# 4.1 .query() : เขียนเงื่อนไขเหมือน SQL
# อ่านง่ายมาก ไม่ต้องพิมพ์ชื่อ df ซ้ำๆ
it_staff = data.query("Department == 'IT' and Salary > 65000")
it_staff

In [ ]:
# 4.2 .isin() : เช็คว่าอยู่ใน List ที่กำหนดไหม
target_depts = ['HR', 'Finance']
support_staff = data[data['Department'].isin(target_depts)]
support_staff

In [ ]:
# 4.3 select_dtypes() : เลือก column ตามชนิดข้อมูล
# เลือกเฉพาะตัวเลข (เช่นเพื่อไปหา Correlation)
numeric_cols = data.select_dtypes(include=['number'])
numeric_cols

## Part 5: การจัดการข้อมูลที่หายไป (Missing Data / NaN)

ข้อมูลที่หายไป (Missing values) พบได้ทั่วไปในข้อมูลจริง จึงต้องเข้าใจวิธีจัดการอย่างเหมาะสม

In [ ]:
df_missing = pd.DataFrame({
    'A': [1, 2, np.nan, 4],
    'B': [5, np.nan, np.nan, 8],
    'C': ['cat', 'dog', 'bird', None]
})

# 5.1 ตรวจสอบ Missing Data
print("Missing per column:\n", df_missing.isnull().sum())

In [ ]:
# 5.2 Fillna (เติมค่า)
# เติม 0 หรือ mean หรือ median
df_missing['A_filled'] = df_missing['A'].fillna(0)

# 5.3 Dropna (ลบทิ้ง)
# thresh=2 คือต้องมีค่าที่ไม่ใช่ NaN อย่างน้อย 2 ตัว ถึงจะไม่ถูกลบ
df_dropped = df_missing.dropna(thresh=2)
df_dropped

## Part 6: การปรับโครงสร้างข้อมูล (Reshaping & Tidy Data: Stack/Unstack)

นอกเหนือจาก Melt/Pivot ควรรู้จัก Stack/Unstack สำหรับจัดการ MultiIndex (Index ที่ซ้อนกันหลายระดับ)

In [ ]:
# สร้าง MultiIndex DataFrame
arrays = [['Bar', 'Bar', 'Baz', 'Baz'], ['one', 'two', 'one', 'two']]
tuples = list(zip(*arrays))
index = pd.MultiIndex.from_tuples(tuples, names=['first', 'second'])
df_multi = pd.DataFrame(np.random.randn(4, 2), index=index, columns=['A', 'B'])
df_multi

In [ ]:
# 6.1 Stack : ดึง Column ลงมาเป็น Index (Table สูงขึ้น)
stacked = df_multi.stack()
print("Stacked:\n", stacked)

# 6.2 Unstack : ผลัก Index ไปเป็น Column (Table กว้างขึ้น)
unstacked = stacked.unstack()
print("\nUnstacked (Back to normal):\n", unstacked)

## Part 7: โจทย์ทำความสะอาดข้อมูลจริง (Real World Cleaning Challenge)

โจทย์นี้จำลองบันทึก (Log) การสั่งซื้อสินค้าจากระบบ E-commerce จริง ซึ่งมีความซับซ้อนและมีข้อมูลไม่สะอาดจำนวนมาก

**ปัญหาที่พบในข้อมูล:**
1.  **Nested Data:** ข้อมูลบางคอลัมน์ถูกเก็บในรูปแบบ String ที่ไม่เป็นมาตรฐาน (คล้าย JSON แต่ไม่ใช่ JSON)
2.  **Inconsistent Dates:** วันที่มีหลายรูปแบบ
3.  **Dirty Strings:** มีอักขระที่ไม่ต้องการปะปนในข้อความ
4.  **Mixed Types:** ตัวเลขและตัวอักษรปะปนกัน

**เป้าหมาย:** แปลงข้อมูลชุดนี้ให้เป็นรายงานสรุปยอดขายรายเมือง

In [ ]:
import io

# ข้อมูลดิบที่สกปรกระดับ Boss (มี Dictionary ที่เก็บเป็น String, Date มั่ว, ข้อมูลแฝง)
boss_data = """
Log_ID|Timestamp|User_Agent|Event_Details|Transaction_Value
LOG-001|2024-01-01 10:30:00|Mozilla/5.0...|{Action: Purchase, Item: Laptop, City: Bangkok]|25,000 THB
LOG-002|01/02/2024 2PM|Safari iPhone|{Action: View, Item: Mouse, City: Chiang Mai]|-
ERROR|Unknown|Bot Spider|{Error: Timeout}|0
LOG-003|2024.01.03 15:45|Chrome Windows|{Action: Purchase, Item: Monitor, City: BANGKOK}|8,500.50
Log-004|Jan 4, 2024 09:00|Edge|{Action: Purchase Item: Keyboard City: PhukeT}|2,000
"""

df_boss = pd.read_csv(io.StringIO(boss_data), sep='|')
df_boss

### Step 1: Filter & Clean Structure
กำจัดแถวที่มีข้อผิดพลาด (Error) และปรับชื่อคอลัมน์ให้เป็นมาตรฐาน

In [ ]:
# 1. ลบช่องว่างหัวท้ายชื่อ Column
df_boss.columns = df_boss.columns.str.strip()

# 2. กรองเฉพาะ Log ที่ถูกต้อง (ขึ้นต้นด้วย LOG แบบ case-insensitive)
# (?i) คือ flag regex ให้ไม่สนใจตัวพิมพ์เล็ก/ใหญ่
df_clean = df_boss[df_boss['Log_ID'].str.contains(r'(?i)^log-', na=False)].copy()
df_clean

### Step 2: Advanced Regex Extraction
ดึงข้อมูล `Action`, `Item` และ `City` ออกจากคอลัมน์ `Event_Details` ซึ่งมีรูปแบบการเขียนไม่สม่ำเสมอ

**การวิเคราะห์รูปแบบ (Pattern Analysis):**
- บางรายการใช้ `:` คั่น แต่รายการสุดท้ายไม่มี
- บางรายการใช้ `,` คั่นกลาง
- `City` อยู่ในตำแหน่งท้ายสุดเสมอ

In [ ]:
# ใช้ Regex ที่ยืดหยุ่น (Flexible)
# Action: (\w+) -> หาคำหลัง Action:
# .*Item: (\w+) -> หาคำหลัง Item: (.* คือข้ามอะไรก็ได้ตรงกลาง)
# .*City: (\w+) -> หาคำหลัง City:

pattern = r'Action:\s*(\w+).*Item:\s*(\w+).*City:\s*(\w+)'
extracted_data = df_clean['Event_Details'].str.extract(pattern, flags=re.IGNORECASE)
extracted_data.columns = ['Action', 'Item', 'City']

# เอาไปแปะกลับในตารางหลัก
df_clean = pd.concat([df_clean, extracted_data], axis=1)
df_clean

### Step 3: Normalize Text (City Name)
ปรับชื่อเมืองให้อยู่ในรูปแบบเดียวกัน เช่น Bangkok และ BANGKOK รวมถึงแก้ไข PhukeT

In [ ]:
# ทำเป็น Title Case (ตัวแรกใหญ่ ที่เหลือเล็ก)
df_clean['City'] = df_clean['City'].str.title()
df_clean

### Step 4: Clean Money
ค่าตัวเลขมีทั้งแบบที่มีหน่วย THB แบบที่มีเครื่องหมาย comma และแบบที่ไม่มีหน่วย

In [ ]:
# ลบทุกอย่างที่ไม่ใช่ตัวเลข หรือจุดทศนิยม
# [^\d.] หมายถึง "ไม่เอาตัวเลขและจุด"
df_clean['Amount'] = pd.to_numeric(
    df_clean['Transaction_Value'].astype(str).str.replace(r'[^\d.]', '', regex=True),
    errors='coerce'
)
df_clean

### Step 5: Final Report
รวมยอดขาย (Purchase) แยกตามเมือง

In [ ]:
# เลือกเฉพาะ Action = Purchase
purchase_only = df_clean.query("Action == 'Purchase'")

# Groupby City
report = purchase_only.groupby('City')['Amount'].sum().reset_index()
report